# LangGraph Retries — Automatic Node Retry with `RetryPolicy`

This notebook demonstrates LangGraph's built-in **node-level retry** support:
attach a `RetryPolicy` to a node, and LangGraph re-runs that node on its own
whenever it raises a matching exception — no manual `try`/`except`/loop code
needed.

Flow: a single node calls a **simulated flaky service** that fails with a
`ConnectionError` on its first two calls and succeeds on the third. The graph
retries it automatically until it succeeds (or `max_attempts` runs out).

**No model call needed** — this is a pure control-flow feature of the graph
runtime, so the whole example runs locally with plain Python.


In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy


**State** — tracks how many attempts the service call took and its final result.

In [ ]:
class CallState(TypedDict):

    attempts: int
    result: str


**Simulated flaky service** — `call_count` stands in for a real service that
is unreliable the first couple of times (a network blip, a cold-starting API,
etc.). The node raises `ConnectionError` until the third call, then succeeds.
`ConnectionError` is one of the exception types LangGraph retries by default.

In [ ]:
call_count = {"n": 0}


def call_flaky_service(state: CallState):

    call_count["n"] += 1
    print(f"Attempt {call_count['n']}: calling the service...")

    if call_count["n"] < 3:
        raise ConnectionError("Service temporarily unavailable")

    return {"attempts": call_count["n"], "result": "Service responded successfully"}


**Attach a `RetryPolicy` to the node** — this is the whole feature. LangGraph
re-invokes `call_flaky_service` on failure, waiting `initial_interval` seconds
and backing off by `backoff_factor` each time, up to `max_attempts` tries
total, before giving up and letting the exception propagate.

In [ ]:
graph_builder = StateGraph(CallState)

graph_builder.add_node(
    "call_service",
    call_flaky_service,
    retry_policy=RetryPolicy(max_attempts=5, initial_interval=0.1, backoff_factor=1.0),
)

graph_builder.add_edge(START, "call_service")
graph_builder.add_edge("call_service", END)

graph = graph_builder.compile()


**Run it.** The node fails twice and the printed attempt log shows LangGraph
retrying it automatically — the caller only ever sees the final, successful
result.

In [1]:
result = graph.invoke({"attempts": 0, "result": ""})
print(result)


Attempt 1: calling the service...
Attempt 2: calling the service...
Attempt 3: calling the service...
{'attempts': 3, 'result': 'Service responded successfully'}
